## Punzi figure of merit

use Punzi (2003) instead of accuracy to pick a BDT threshold that actually minimises experiment runtime.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import pickle
import os

os.makedirs("../plots", exist_ok=True)

In [ ]:
features = ["PT1", "PT2", "P1", "P2", "TotalPT", "VertexChisq", "Isolation", "MASS"]
features7 = [f for f in features if f != "MASS"]

signal = pd.read_csv("../data/signal_Bs2MuMu.txt", sep=r"\s+", header=None, names=features)
background = pd.read_csv("../data/background_combinatorial.txt", sep=r"\s+", header=None, names=features)
signal = signal.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
background = background.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

with open("../bdt_model.pkl", "rb") as fh:
    bdt7 = pickle.load(fh)
with open("../bdt_results.json") as fh:
    bdt_results = json.load(fh)

In [ ]:
scores_sig = bdt7.predict_proba(signal[features7])[:, 1]
scores_bkg = bdt7.predict_proba(background[features7])[:, 1]

N_BKG_YEAR = 2000
n_sigma = 5.0
bkg_eff_floor = 1.0 / len(background)   # 10k MC events -> can't measure below this

thresholds = np.linspace(0.01, 0.99, 300)
fom_vals, acc_vals, sig_eff_scan, bkg_eff_scan = [], [], [], []
for t in thresholds:
    eps_s = (scores_sig >= t).mean()
    eps_b = max((scores_bkg >= t).mean(), bkg_eff_floor)
    B = N_BKG_YEAR * eps_b
    fom_vals.append(eps_s / (n_sigma / 2 + np.sqrt(B)))
    acc_vals.append(((scores_sig >= t).sum() + (scores_bkg < t).sum()) / (len(scores_sig) + len(scores_bkg)))
    sig_eff_scan.append(eps_s)
    bkg_eff_scan.append(eps_b)

fom_vals, acc_vals = np.array(fom_vals), np.array(acc_vals)
sig_eff_scan, bkg_eff_scan = np.array(sig_eff_scan), np.array(bkg_eff_scan)

i_fom, i_acc = int(np.argmax(fom_vals)), int(np.argmax(acc_vals))
sig_eff_punzi, bkg_eff_punzi = sig_eff_scan[i_fom], bkg_eff_scan[i_fom]

print(f"Punzi:    t = {thresholds[i_fom]:.3f}, sig_eff = {sig_eff_punzi:.4f}, bkg_eff = {bkg_eff_punzi:.4f}")
print(f"Accuracy: t = {thresholds[i_acc]:.3f}, sig_eff = {sig_eff_scan[i_acc]:.4f}, bkg_eff = {bkg_eff_scan[i_acc]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax2 = ax.twinx()
ax.plot(thresholds, fom_vals, color="steelblue", label="Punzi FOM")
ax2.plot(thresholds, acc_vals, color="darkorange", linestyle="--", label="accuracy")
ax.axvline(thresholds[i_fom], color="steelblue", linestyle=":", alpha=0.8)
ax.axvline(thresholds[i_acc], color="darkorange", linestyle=":", alpha=0.8)
ax.set_xlabel("BDT threshold"); ax.set_ylabel("Punzi FOM", color="steelblue")
ax2.set_ylabel("accuracy", color="darkorange")

ax = axes[1]
ax.plot(thresholds, sig_eff_scan, label="signal")
ax.plot(thresholds, bkg_eff_scan, label="background")
ax.axvline(thresholds[i_fom], linestyle="--", color="black", label="Punzi")
ax.set_xlabel("BDT threshold"); ax.set_ylabel("efficiency"); ax.legend()

plt.tight_layout()
plt.savefig("../plots/punzi_fom.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# S/sqrt(B) drives discovery time (T ~ 1 / (S/sqrt(B))^2)
sens_punzi = (50 * sig_eff_punzi) / np.sqrt(N_BKG_YEAR * bkg_eff_punzi)
sens_acc = (50 * sig_eff_scan[i_acc]) / np.sqrt(N_BKG_YEAR * bkg_eff_scan[i_acc])
print(f"S/sqrt(B): Punzi = {sens_punzi:.3f}, accuracy = {sens_acc:.3f}, gain = {sens_punzi/sens_acc:.3f}x")

bdt_results["signal_efficiency_punzi"] = float(sig_eff_punzi)
bdt_results["background_efficiency_punzi"] = float(bkg_eff_punzi)
with open("../bdt_results.json", "w") as fh:
    json.dump(bdt_results, fh, indent=2)